# Karachi AQI EDA

Exploratory analysis for the Pearls AQI Predictor feature set. This notebook loads the same Hopsworks/local feature data used by the training pipeline and summarizes AQI distribution, pollutant drivers, weather missingness, and time patterns in Karachi local time.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dotenv import load_dotenv
import pandas as pd

from aqi_predictor.config.settings import get_settings
from aqi_predictor.services.storage import FEATURES_FEATURE_GROUP, FeatureStoreClient

load_dotenv()
settings = get_settings()
settings

In [ ]:
store = FeatureStoreClient(settings)
features = store.read_feature_group(FEATURES_FEATURE_GROUP)
if features.empty:
    features = pd.read_csv(ROOT / "data/processed/karachi_aqi_features.csv")
features["event_time"] = pd.to_datetime(features["event_time"], utc=True)
features = features.sort_values("event_time")
features.shape

In [ ]:
features[["event_time", "city", "aqi_score", "aqi_category", "primary_pollutant", "pm2_5", "pm10"]].tail()

In [ ]:
features["aqi_score"].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).to_frame("aqi_score")

In [ ]:
features["aqi_category"].value_counts(dropna=False).to_frame("hours")

In [ ]:
pollutants = ["pm2_5", "pm10", "co", "no2", "o3", "so2", "nh3"]
weather = ["temp", "feels_like", "pressure", "humidity", "wind_speed", "wind_deg", "clouds"]
features[pollutants].describe().T[["mean", "std", "min", "50%", "max"]]

In [ ]:
features[weather].isna().mean().mul(100).sort_values(ascending=False).to_frame("missing_pct")

In [ ]:
local_time = features["event_time"].dt.tz_convert(settings.timezone)
features.assign(hour=local_time.dt.hour).groupby("hour")["aqi_score"].agg(["mean", "min", "max", "count"])

In [ ]:
candidate_columns = pollutants + ["aqi_lag_1h", "aqi_rolling_3h", "aqi_rolling_24h", "aqi_change_rate", "hour", "weekday"]
available = [column for column in candidate_columns if column in features.columns]
features[available + ["aqi_score"]].corr(numeric_only=True)["aqi_score"].drop("aqi_score").sort_values(key=lambda s: s.abs(), ascending=False)